# Predicción del riesgo de estrés financiero en asalariados de la Comunidad de Madrid
## TFM — Máster de Data Science
### Pipeline de datos: Capa Gold
---

**Input:** `data/02_silver/dataset_analitico.csv` — 2.947 asalariados, 64 variables limpias  
**Output:** `data/03_gold/dataset_modelado.csv` — dataset listo para modelado (features + target)

---

### Estructura del notebook

| Sección | Contenido |
|---------|-----------|
| G.1 | Carga y auditoría del dataset Silver |
| G.2 | Construcción del target `estres_financiero_alto` |
| G.3 | Eliminación de variables no útiles para el modelo |
| G.4 | Feature engineering |
| G.5 | Tratamiento de nulos residuales |
| G.6 | Encoding de variables categóricas |
| G.7 | Resumen del dataset Gold y exportación |
| G.8 | Decisiones de diseño — resumen |

> **Nota:** Este notebook documenta y justifica cada decisión con exploración visual. El código de producción ejecutable está en `gold_ecv.py`.


## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.family': 'sans-serif',
})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
sns.set_palette(PALETTE)

PATH_SILVER = Path('data/02_silver/dataset_analitico.csv')
PATH_GOLD   = Path('data/03_gold/dataset_modelado.csv')

df = pd.read_csv(PATH_SILVER, low_memory=False)
print(f"Dataset Silver cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")


---
## G.1 Auditoría del dataset Silver

Antes de cualquier transformación Gold se revisa el estado heredado del pipeline Silver: tipos de datos, nulos y variables que ya pueden descartarse por ser constantes o redundantes.


In [ ]:
# Variables constantes (un único valor en toda la muestra)
constantes = [col for col in df.columns if df[col].nunique(dropna=True) <= 1]
print("Variables constantes (se eliminan en G.3):")
for c in constantes:
    print(f"  {c}: {df[c].unique().tolist()}")


In [ ]:
# Verificar que renta_neta_hogar y renta_hogar_indicadores son idénticas
corr = df[['renta_neta_hogar', 'renta_hogar_indicadores']].corr().iloc[0, 1]
dif  = (df['renta_neta_hogar'] - df['renta_hogar_indicadores']).abs().max()
print(f"Correlación renta_neta_hogar vs renta_hogar_indicadores: {corr:.6f}")
print(f"Diferencia máxima entre ambas:                           {dif:.6f}")
print()
print("→ Son idénticas. Se elimina renta_hogar_indicadores (redundante).")


In [ ]:
# Resumen de nulos heredados del Silver
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
resumen_nulos = pd.DataFrame({'n_nulos': nulos, '% nulos': nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos.n_nulos > 0].sort_values('n_nulos', ascending=False)

print(f"Variables con nulos: {len(resumen_nulos)} de {df.shape[1]}")
print()
print(resumen_nulos.to_string())


---
## G.2 Construcción del target `estres_financiero_alto`

### Definición

El target es una **variable binaria** que indica estrés financiero alto. Se construye a partir de los 5 componentes disponibles en la ECV, adoptando una regla de **mayoría simple: al menos 2 de 5 condiciones activas**.

| Componente | Variable | Condición de estrés |
|------------|----------|---------------------|
| Dificultad fin de mes | `capacidad_fin_de_mes` | `'Con mucha dificultad'` o `'Con dificultad'` |
| Sin colchón para imprevistos | `capacidad_gastos_imprevistos` | `'No (no puede)'` |
| Retrasos en facturas | `retrasos_facturas` | `'Sí, una vez'` o `'Sí, dos o más veces'` |
| Retrasos hipoteca/alquiler | `retrasos_hipoteca_alquiler` | `'Sí, una vez'` o `'Sí, dos o más veces'` |
| Retrasos en deudas | `retrasos_deudas_no_vivienda` | `'Sí, una vez'` o `'Sí, dos o más veces'` |

**¿Por qué ≥ 2 condiciones?** Un único indicador puede reflejar una situación puntual o excepcional. Exigir al menos dos condiciones simultáneas garantiza que el perfil de estrés sea **persistente y multidimensional**, coherente con la literatura sobre vulnerabilidad financiera de hogares.

**Tratamiento de NaN en los componentes:** Los 5 componentes tienen nulos mínimos (≤16 observaciones). Las filas donde algún componente es NaN se evalúan sobre los componentes disponibles; si con los disponibles ya se alcanzan ≥2 condiciones activas, se clasifica como estrés alto igualmente. Si no, se conserva como NaN en el target para no introducir información falsa.


In [ ]:
COMPONENTES = {
    'capacidad_fin_de_mes':         ['Con mucha dificultad', 'Con dificultad'],
    'capacidad_gastos_imprevistos': ['No (no puede)'],
    'retrasos_facturas':            ['Sí, una vez', 'Sí, dos o más veces'],
    'retrasos_hipoteca_alquiler':   ['Sí, una vez', 'Sí, dos o más veces'],
    'retrasos_deudas_no_vivienda':  ['Sí, una vez', 'Sí, dos o más veces'],
}

# Distribución individual de cada componente
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Distribución individual de los 5 componentes del estrés financiero',
             fontweight='bold', y=1.01)

COLORES_COMP = {'stress': '#C73E1D', 'ok': '#2E86AB', 'nan': '#999'}

for ax, (col, vals_stress) in zip(axes, COMPONENTES.items()):
    vc = df[col].value_counts(dropna=False)
    colores = []
    for k in vc.index:
        if pd.isna(k):         colores.append(COLORES_COMP['nan'])
        elif k in vals_stress: colores.append(COLORES_COMP['stress'])
        else:                  colores.append(COLORES_COMP['ok'])
    bars = ax.barh([str(k) for k in vc.index], vc.values, color=colores)
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=8)
    pct_stress = df[col].isin(vals_stress).mean() * 100
    ax.set_title(col.replace('_', '
'), fontsize=8, fontweight='bold')
    ax.set_xlabel(f'n  ({pct_stress:.1f}% estrés)', fontsize=8)
    ax.set_xlim(0, vc.max() * 1.2)

plt.tight_layout()
plt.savefig('gold_componentes_target.png', bbox_inches='tight')
plt.show()
print("Rojo = condición de estrés activa | Azul = sin estrés | Gris = NaN")


In [ ]:
# Construir score (0-5) y target binario
for col, vals in COMPONENTES.items():
    df[f'_comp_{col}'] = df[col].isin(vals).astype('Int64')

comp_cols = [f'_comp_{c}' for c in COMPONENTES]
df['_score_estres'] = df[comp_cols].sum(axis=1, skipna=True)

# Target: ≥2 condiciones activas
# Si el score es ≥2 → 1; si es 0 o 1 → 0; si hay NaN que impiden decidir → NaN
n_comp_disponibles = df[comp_cols].notna().sum(axis=1)
df['estres_financiero_alto'] = np.where(
    df['_score_estres'] >= 2, 1,
    np.where(
        (df['_score_estres'] < 2) & (n_comp_disponibles >= 4), 0,
        np.nan
    )
).astype('Int64')

# Limpiar columnas auxiliares
df = df.drop(columns=comp_cols + ['_score_estres'])

# Distribución del target
vc = df['estres_financiero_alto'].value_counts(dropna=False)
n_nan = df['estres_financiero_alto'].isna().sum()
n0 = vc.get(0, 0)
n1 = vc.get(1, 0)
print(f"TARGET: estres_financiero_alto")
print(f"  0 — sin estrés alto:  {n0:,}  ({n0/len(df)*100:.1f}%)")
print(f"  1 — estrés alto:      {n1:,}  ({n1/len(df)*100:.1f}%)")
print(f"  NaN (indeterminado):  {n_nan:,}  ({n_nan/len(df)*100:.1f}%)")
print(f"  Ratio desbalanceo:    1:{n0/n1:.1f}  (clase 0 vs clase 1)")


In [ ]:
# Visualización: score y target
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Score 0-5
score_vc = df['_score_estres'].value_counts().sort_index() if '_score_estres' in df.columns else     pd.Series({0:1982, 1:500, 2:288, 3:100, 4:55, 5:22})

# Reconstruir score para el plot
for col, vals in COMPONENTES.items():
    df[f'_comp_{col}'] = df[col].isin(vals).astype('Int64')
comp_cols = [f'_comp_{c}' for c in COMPONENTES]
df['_score_plot'] = df[comp_cols].sum(axis=1, skipna=True)
df = df.drop(columns=comp_cols)

score_vc = df['_score_plot'].value_counts().sort_index()
colores_score = ['#2E86AB' if i < 2 else '#C73E1D' for i in score_vc.index]
bars0 = axes[0].bar(score_vc.index.astype(str), score_vc.values, color=colores_score, width=0.6)
axes[0].bar_label(bars0, fmt='%d', padding=3, fontsize=9)
axes[0].axvline(1.5, color='gray', linestyle='--', alpha=0.7, label='Umbral ≥2')
axes[0].set_xlabel('Número de condiciones de estrés activas')
axes[0].set_ylabel('Personas')
axes[0].set_title('Distribución del score de estrés (0–5)
Azul = sin estrés alto | Rojo = estrés alto', fontweight='bold')
axes[0].legend(fontsize=9)
df = df.drop(columns=['_score_plot'])

# Target binario
target_vc = df['estres_financiero_alto'].value_counts()
etiquetas = [f'0 — Sin estrés
({target_vc.get(0,0)/len(df)*100:.1f}%)',
             f'1 — Estrés alto
({target_vc.get(1,0)/len(df)*100:.1f}%)']
axes[1].pie([target_vc.get(0,0), target_vc.get(1,0)],
            labels=etiquetas, colors=['#2E86AB', '#C73E1D'],
            autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target binario: estres_financiero_alto
(umbral: ≥2 condiciones)', fontweight='bold')

plt.tight_layout()
plt.savefig('gold_target_distribucion.png', bbox_inches='tight')
plt.show()

print(f"⚠️  Desbalanceo 1:{target_vc.get(0,0)/target_vc.get(1,0):.1f}.")
print("   Estrategia recomendada en modelado: class_weight='balanced' o SMOTE.")


---
## G.3 Eliminación de variables no útiles para el modelo

Se eliminan cuatro grupos de variables:

1. **Constantes** — no aportan varianza al modelo: `region`, `situacion_actividad`, `situacion_profesional`
2. **Identificadores** — no son features: `id_hogar`, `id_persona`
3. **Componentes del target** — se han consumido para construir `estres_financiero_alto`; incluirlas como features produciría *data leakage* directo
4. **Variable duplicada** — `renta_hogar_indicadores` es idéntica a `renta_neta_hogar`

> **`peso_persona` se conserva** en el dataset Gold como columna auxiliar para permitir estadísticas descriptivas ponderadas, pero **no se incluye como feature** en el modelo.


In [ ]:
COLS_CONSTANTES   = ['region', 'situacion_actividad', 'situacion_profesional']
COLS_IDS          = ['id_hogar', 'id_persona']
COLS_TARGET_LEAK  = list(COMPONENTES.keys())   # los 5 componentes usados para construir el target
COLS_DUPLICADAS   = ['renta_hogar_indicadores']

cols_a_eliminar = COLS_CONSTANTES + COLS_IDS + COLS_TARGET_LEAK + COLS_DUPLICADAS
cols_eliminadas = [c for c in cols_a_eliminar if c in df.columns]
df = df.drop(columns=cols_eliminadas)

print(f"Columnas eliminadas: {len(cols_eliminadas)}")
for grp, cols in [
    ('Constantes',          COLS_CONSTANTES),
    ('Identificadores',     COLS_IDS),
    ('Componentes target',  COLS_TARGET_LEAK),
    ('Duplicadas',          COLS_DUPLICADAS),
]:
    print(f"  {grp}: {[c for c in cols if c in cols_eliminadas]}")

print(f"\nColumnas restantes: {df.shape[1]}  ({df.shape[0]:,} filas)")


---
## G.4 Feature engineering

Se construyen variables derivadas que capturan relaciones económicas relevantes no disponibles directamente en la ECV.


### G.4.1 Renta neta per cápita del hogar

La renta del hogar en bruto no es comparable entre hogares de distinto tamaño. La renta per cápita ajustada por unidades de consumo (escala OCDE modificada) es el indicador estándar del INE para comparar el nivel de vida.

$$\text{renta\_hogar\_per\_capita} = \frac{\text{renta\_neta\_hogar}}{\text{unidades\_consumo}}$$


In [ ]:
df['renta_hogar_per_capita'] = df['renta_neta_hogar'] / df['unidades_consumo']

print("renta_hogar_per_capita — estadísticas:")
print(df['renta_hogar_per_capita'].describe(percentiles=[.1,.25,.5,.75,.9,.99]).round(0))


### G.4.2 Ratio de carga hipotecaria/alquiler sobre renta

El porcentaje que representan los gastos de vivienda sobre la renta es un indicador clave de presión financiera. Un ratio > 30% se considera sobrecarga según los estándares europeos (Eurostat).


In [ ]:
# gastos_vivienda incluye alquiler O cuota hipoteca (según régimen de tenencia)
# Se usa renta_neta_salarial como denominador (renta propia del asalariado)
# Protegemos la división por cero o renta nula con np.where
df['ratio_carga_vivienda'] = np.where(
    df['renta_neta_salarial'] > 0,
    (df['gastos_vivienda'] * 12) / df['renta_neta_salarial'],
    np.nan
)

# Capear al percentil 99 para evitar que rentas muy bajas distorsionen
p99 = df['ratio_carga_vivienda'].quantile(0.99)
df['ratio_carga_vivienda'] = df['ratio_carga_vivienda'].clip(upper=p99)

fig, ax = plt.subplots(figsize=(9, 3.5))
df['ratio_carga_vivienda'].dropna().hist(bins=50, ax=ax, color='#2E86AB', alpha=0.8, edgecolor='white')
ax.axvline(0.30, color='#C73E1D', linestyle='--', linewidth=1.5, label='30% (umbral sobrecarga)')
ax.set_xlabel('Ratio gastos vivienda / renta salarial neta')
ax.set_ylabel('Personas')
ax.set_title('Distribución del ratio de carga de vivienda', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('gold_ratio_vivienda.png', bbox_inches='tight')
plt.show()

n_sobrecarga = (df['ratio_carga_vivienda'] > 0.30).sum()
print(f"Personas con ratio > 30% (sobrecarga): {n_sobrecarga} ({n_sobrecarga/len(df)*100:.1f}%)")


### G.4.3 Indicador de precariedad laboral

Se construye un indicador binario que combina dos dimensiones de precariedad: tipo de contrato temporal y jornada a tiempo parcial. La combinación de ambas es la situación de mayor vulnerabilidad laboral.


In [ ]:
# Temporal = cualquier modalidad temporal (escrito o verbal)
es_temporal    = df['tipo_contrato'].isin(['Temporal escrito', 'Temporal verbal'])
es_parcial     = df['jornada'] == 'Tiempo parcial'

df['precariedad_laboral'] = (es_temporal | es_parcial).astype('Int64')

# Desglose
print("Desglose del indicador de precariedad laboral:")
print(f"  Solo temporal (contrato):  {(es_temporal & ~es_parcial).sum():>5}")
print(f"  Solo parcial (jornada):    {(~es_temporal & es_parcial).sum():>5}")
print(f"  Temporal + parcial:        {(es_temporal & es_parcial).sum():>5}")
print(f"  ─────────────────────────────────")
print(f"  Total precarios:           {df['precariedad_laboral'].sum():>5} ({df['precariedad_laboral'].mean()*100:.1f}%)")
print(f"  NaN (componente nulo):     {df['precariedad_laboral'].isna().sum():>5}")


### G.4.4 Agrupación de `nivel_estudios`

El dataset Silver tiene 7 categorías de nivel educativo, con dos grupos muy minoritarios (`'Sin estudios'` n=16, `'Secundaria 2ª etapa (gral)'` n=11) que generarían inestabilidad en los modelos. Se consolidan en grupos más robustos.

| Grupo resultante | Categorías originales |
|------------------|-----------------------|
| Hasta primaria | Sin estudios + Primaria incompleta + Primaria |
| Secundaria 1ª etapa | Secundaria 1ª etapa + Secundaria 1ª etapa (título) |
| Post-secundaria | Post-secundaria no superior + Secundaria 2ª etapa (gral) |


In [ ]:
# Distribución original
print("Distribución original de nivel_estudios:")
print(df['nivel_estudios'].value_counts(dropna=False).to_string())

mapa_estudios = {
    'Sin estudios':                    'Hasta primaria',
    'Primaria incompleta':             'Hasta primaria',
    'Primaria':                        'Hasta primaria',
    'Secundaria 1ª etapa':             'Secundaria 1ª etapa',
    'Secundaria 1ª etapa (título)':    'Secundaria 1ª etapa',
    'Secundaria 2ª etapa (gral)':      'Post-secundaria',
    'Post-secundaria no superior':     'Post-secundaria',
}
df['nivel_estudios'] = df['nivel_estudios'].map(mapa_estudios)

print("\nDistribución tras agrupación:")
print(df['nivel_estudios'].value_counts(dropna=False).to_string())


### G.4.5 Transformación logarítmica de rentas con alta asimetría

Las variables de renta presentan fuerte asimetría positiva (skew > 2.5), lo que perjudica a modelos lineales y basados en distancias. Se aplica `log1p` (log(1+x)) para estabilizar la varianza y acercar la distribución a la normalidad. Se usa `log1p` en lugar de `log` para manejar correctamente los ceros.


In [ ]:
COLS_LOG1P = [
    'renta_neta_salarial',
    'renta_no_monetaria_salarial',
    'renta_neta_hogar',
    'renta_hogar_per_capita',
    'importe_alquiler',
    'cuota_hipoteca',
    'gastos_vivienda',
]

fig, axes = plt.subplots(2, len(COLS_LOG1P), figsize=(18, 6))
fig.suptitle('Efecto de log1p sobre las distribuciones de renta y vivienda', fontweight='bold')

for i, col in enumerate(COLS_LOG1P):
    if col not in df.columns:
        continue
    datos = df[col].dropna()
    axes[0, i].hist(datos, bins=40, color='#2E86AB', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{col}\nskew={datos.skew():.2f}', fontsize=7)
    axes[0, i].set_ylabel('n' if i == 0 else '')

    datos_log = np.log1p(datos)
    axes[1, i].hist(datos_log, bins=40, color='#A23B72', alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'log1p({col})\nskew={datos_log.skew():.2f}', fontsize=7)
    axes[1, i].set_ylabel('n' if i == 0 else '')

axes[0, 0].text(-0.25, 0.5, 'Original', transform=axes[0,0].transAxes,
                rotation=90, va='center', fontweight='bold', fontsize=9)
axes[1, 0].text(-0.25, 0.5, 'log1p', transform=axes[1,0].transAxes,
                rotation=90, va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('gold_log1p_rentas.png', bbox_inches='tight')
plt.show()

# Aplicar transformación
for col in COLS_LOG1P:
    if col in df.columns:
        nombre_log = f'log_{col}'
        df[nombre_log] = np.log1p(df[col].clip(lower=0))

print(f"Variables log1p creadas: {[f'log_{c}' for c in COLS_LOG1P if c in df.columns]}")
print("Las columnas originales se conservan para referencia y estadísticas descriptivas.")


---
## G.5 Tratamiento de nulos residuales

Tras la capa Silver quedan nulos en dos grupos con estrategias distintas:

**Grupo I — Nulos informativos (pequeño volumen, < 2%)**  
Son datos que deberían existir pero no se recogieron. Se imputan con la mediana (numéricas) o la moda (categóricas) calculadas sobre el conjunto de entrenamiento. En este notebook se imputan sobre el dataset completo a efectos de documentación; en producción la imputación se ajusta solo sobre train.

**Grupo II — Nulos estructurales persistentes (> 2%)**  
`motivo_aumento_ingresos` (82.7%) y `motivo_disminucion_ingresos` (90.4%) son NaN porque la pregunta no aplica cuando los ingresos no cambiaron. Se crean variables indicadoras binarias y se imputa la categoría `'No aplica'`.

`expectativa_ingresos_12m` (9.8%) es un módulo de respuesta opcional; se imputa con la moda y se crea un indicador de no-respuesta.


In [ ]:
# ── Estado de nulos antes de imputación ──────────────────────────────────────
nulos_pre = df.isnull().sum()
nulos_pre = nulos_pre[nulos_pre > 0].sort_values(ascending=False)
print("Nulos antes de imputación Gold:")
for col, n in nulos_pre.items():
    print(f"  {col:<45} {n:>5}  ({n/len(df)*100:.1f}%)")


In [ ]:
# ── Grupo II: nulos estructurales persistentes ────────────────────────────────

# motivo_aumento_ingresos y motivo_disminucion_ingresos:
# NaN significa "no hubo cambio" → se imputa como categoría explícita
df['motivo_aumento_ingresos']      = df['motivo_aumento_ingresos'].fillna('No aplica (sin aumento)')
df['motivo_disminucion_ingresos']  = df['motivo_disminucion_ingresos'].fillna('No aplica (sin disminución)')

# expectativa_ingresos_12m: módulo opcional (9.8% NaN)
# Se crea indicador de no-respuesta y se imputa con la moda
df['expectativa_sin_respuesta'] = df['expectativa_ingresos_12m'].isna().astype(int)
moda_expectativa = df['expectativa_ingresos_12m'].mode()[0]
df['expectativa_ingresos_12m']  = df['expectativa_ingresos_12m'].fillna(moda_expectativa)

print(f"motivo_aumento_ingresos — nulos restantes:      {df['motivo_aumento_ingresos'].isna().sum()}")
print(f"motivo_disminucion_ingresos — nulos restantes:  {df['motivo_disminucion_ingresos'].isna().sum()}")
print(f"expectativa_ingresos_12m — nulos restantes:     {df['expectativa_ingresos_12m'].isna().sum()}")
print(f"expectativa_sin_respuesta — n personas:         {df['expectativa_sin_respuesta'].sum()}")


In [ ]:
# ── Grupo I: nulos informativos (<2%) ─────────────────────────────────────────
# Numéricas → mediana | Categóricas → moda
# Nota: en producción calcular mediana/moda solo sobre train set

nulos_post_ii = df.isnull().sum()
nulos_post_ii = nulos_post_ii[nulos_post_ii > 0].sort_values(ascending=False)

cols_num = df.select_dtypes(include='number').columns.tolist()
cols_cat = df.select_dtypes(include='object').columns.tolist()

log_imputacion = []
for col in nulos_post_ii.index:
    if col == 'estres_financiero_alto':
        continue  # el target no se imputa
    n = int(df[col].isna().sum())
    if col in cols_num:
        val = df[col].median()
        df[col] = df[col].fillna(val)
        log_imputacion.append({'Variable': col, 'Tipo': 'numérica', 'n imputadas': n, 'Valor': f'mediana={val:.2f}'})
    elif col in cols_cat:
        val = df[col].mode()[0]
        df[col] = df[col].fillna(val)
        log_imputacion.append({'Variable': col, 'Tipo': 'categórica', 'n imputadas': n, 'Valor': f'moda={repr(val)}'})

print("Imputaciones Grupo I (nulos informativos <2%):")
print(pd.DataFrame(log_imputacion).to_string(index=False))
print(f"\nNulos totales restantes: {df.isnull().sum().sum()}")
print("(Solo puede quedar el target si alguna fila era indeterminable)")


---
## G.6 Encoding de variables categóricas

Se aplican tres estrategias según el tipo de variable:

| Estrategia | Variables | Criterio |
|------------|-----------|----------|
| **Ordinal manual** | Variables con orden semántico claro | El orden tiene significado interpretable |
| **Binario (0/1)** | Variables con exactamente 2 categorías | Evita la expansión de columnas innecesaria |
| **One-hot encoding** | Variables nominales sin orden | Sin referencia conceptual entre categorías |

Las variables con orden semántico (estado de salud, dificultad fin de mes, etc.) reciben encoding ordinal porque su orden es información real que el modelo debe poder explotar.


In [ ]:
# ── Encoding ordinal (orden semántico explícito) ───────────────────────────────
ORDINALES = {
    'nivel_estudios': {
        'Hasta primaria': 0, 'Secundaria 1ª etapa': 1, 'Post-secundaria': 2,
    },
    'estado_salud': {
        'Muy malo': 0, 'Malo': 1, 'Regular': 2, 'Bueno': 3, 'Muy bueno': 4,
    },
    'limitacion_actividad': {
        'Gravemente limitado': 0, 'Limitado (no grave)': 1, 'No limitado': 2,
    },
    'grado_urbanizacion': {
        'Zona poco poblada': 0, 'Zona media': 1, 'Zona muy poblada': 2,
    },
    'cambio_ingresos_12m': {
        'Han disminuido': 0, 'Se mantienen': 1, 'Han aumentado': 2,
    },
    'expectativa_ingresos_12m': {
        'Empeorar': 0, 'Mantenerse': 1, 'Mejorar': 2,
    },
    'carga_prestamos_no_vivienda': {
        'Una carga pesada': 0, 'Una carga razonable': 1, 'Ninguna carga': 2,
    },
    'carga_asistencia_medica': {
        'Una carga pesada': 0, 'Una carga razonable': 1, 'Ninguna carga': 2, 'No ha utilizado': 3,
    },
    'carga_asistencia_dental': {
        'Una carga pesada': 0, 'Una carga razonable': 1, 'Ninguna carga': 2, 'No ha utilizado': 3,
    },
    'carga_medicamentos': {
        'Una carga pesada': 0, 'Una carga razonable': 1, 'Ninguna carga': 2, 'No ha consumido': 3,
    },
}

for col, mapa in ORDINALES.items():
    if col in df.columns:
        df[col] = df[col].map(mapa)

print(f"Variables con encoding ordinal: {len(ORDINALES)}")


In [ ]:
# ── Encoding binario (0/1) ────────────────────────────────────────────────────
BINARIAS_SI_NO = [
    'sexo', 'jornada', 'personal_a_cargo', 'enfermedad_cronica',
    'necesito_medico_no_fue', 'puede_vacaciones', 'puede_proteina_2dias',
    'puede_calefaccion_invierno', 'hogar_riesgo_pobreza', 'hogar_carencia_material',
    'arope_2020', 'arope_2030', 'carencia_material_social_severa',
    'baja_intensidad_laboral_2020',
]

MAPA_BINARIO = {
    'Sí': 1, 'No': 0,
    'Hombre': 1, 'Mujer': 0,
    'Tiempo completo': 1, 'Tiempo parcial': 0,
    'No aplicable (≥60 años)': np.nan,  # baja_intensidad_laboral_2020 tiene esta cat
}

for col in BINARIAS_SI_NO:
    if col in df.columns:
        df[col] = df[col].map(MAPA_BINARIO)

print(f"Variables con encoding binario: {len(BINARIAS_SI_NO)}")


In [ ]:
# ── One-hot encoding (nominales sin orden) ────────────────────────────────────
# Se excluyen: columnas ya encodadas, el target, peso_persona, cols log
ya_encodadas = set(ORDINALES.keys()) | set(BINARIAS_SI_NO)
excluir      = ya_encodadas | {'estres_financiero_alto', 'peso_persona',
                                'anio_nacimiento'}  # anio_nacimiento → se usa edad

cols_ohe = [c for c in df.select_dtypes(include='object').columns
            if c not in excluir]

print(f"Variables para one-hot encoding ({len(cols_ohe)}):")
for c in cols_ohe:
    print(f"  {c}: {df[c].nunique(dropna=True)} categorías")

df = pd.get_dummies(df, columns=cols_ohe, drop_first=False, dtype=int)

print(f"\nColumnas tras one-hot: {df.shape[1]}")


In [ ]:
# Eliminar anio_nacimiento (redundante con edad)
if 'anio_nacimiento' in df.columns:
    df = df.drop(columns=['anio_nacimiento'])
    print("anio_nacimiento eliminado (redundante con edad)")

# Resumen tipos tras encoding
print(f"\nShape final: {df.shape}")
print(f"\nDistribución de tipos:")
print(df.dtypes.value_counts().to_string())


---
## G.7 Resumen del dataset Gold y exportación


In [ ]:
# Separar features, target y peso
y       = df['estres_financiero_alto']
peso    = df['peso_persona']
X       = df.drop(columns=['estres_financiero_alto', 'peso_persona'])

print(f"{'═'*55}")
print(f"DATASET GOLD — RESUMEN")
print(f"{'═'*55}")
print(f"  Observaciones:       {len(df):,}")
print(f"  Features (X):        {X.shape[1]}")
print(f"  Target (y):")
print(f"    0 — sin estrés:    {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")
print(f"    1 — estrés alto:   {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)")
print(f"    NaN:               {y.isna().sum():,}")
print(f"  Peso muestral:       conservado (no es feature)")
print(f"  Nulos en X:          {X.isnull().sum().sum():,}")
print(f"{'═'*55}")


In [ ]:
# Exportar dataset completo (X + y + peso)
PATH_GOLD.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PATH_GOLD, index=False, encoding='utf-8-sig')
print(f"Dataset Gold exportado → {PATH_GOLD}")
print(f"  {df.shape[0]:,} filas × {df.shape[1]} columnas")


In [ ]:
# Inventario final de features agrupadas
grupos_gold = {
    'Target': ['estres_financiero_alto'],
    'Peso muestral': ['peso_persona'],
    'Demográficas': ['edad', 'sexo', 'pais_nacimiento', 'nacionalidad'],
    'Situación laboral': ['horas_semana', 'jornada', 'tipo_contrato', 'personal_a_cargo',
                         'anios_experiencia', 'meses_desempleo_ref', 'meses_desempleo_5anios',
                         'precariedad_laboral'],
    'Educación': ['nivel_estudios'],
    'Salud': ['estado_salud', 'enfermedad_cronica', 'limitacion_actividad', 'necesito_medico_no_fue'],
    'Renta (originales)': ['renta_neta_salarial', 'renta_no_monetaria_salarial',
                           'renta_neta_hogar', 'renta_hogar_per_capita',
                           'importe_alquiler', 'cuota_hipoteca', 'gastos_vivienda'],
    'Renta (log1p)': [c for c in df.columns if c.startswith('log_')],
    'Ratio vivienda': ['ratio_carga_vivienda'],
    'Vivienda': ['num_habitaciones', 'unidades_consumo', 'num_miembros_hogar'],
    'Privación material': ['puede_vacaciones', 'puede_proteina_2dias', 'puede_calefaccion_invierno',
                           'puede_sustituir_muebles', 'carga_prestamos_no_vivienda'],
    'Cargas sanitarias': ['carga_asistencia_medica', 'carga_asistencia_dental', 'carga_medicamentos'],
    'Dinámica ingresos': ['cambio_ingresos_12m', 'expectativa_ingresos_12m', 'expectativa_sin_respuesta'],
    'Indicadores pobreza': ['hogar_riesgo_pobreza', 'hogar_carencia_material',
                            'arope_2020', 'arope_2030', 'carencia_material_social_severa',
                            'baja_intensidad_laboral_2020', 'grado_urbanizacion'],
    'OHE — sector CNAE': [c for c in df.columns if c.startswith('sector_cnae_')],
    'OHE — tipo contrato': [c for c in df.columns if c.startswith('tipo_contrato_')],
    'OHE — tipo hogar': [c for c in df.columns if c.startswith('tipo_hogar_')],
    'OHE — tipo vivienda': [c for c in df.columns if c.startswith('tipo_vivienda_')],
    'OHE — régimen tenencia': [c for c in df.columns if c.startswith('regimen_tenencia_')],
    'OHE — pais/nac': [c for c in df.columns if c.startswith('pais_nacimiento_') or c.startswith('nacionalidad_')],
    'OHE — motivos ingresos': [c for c in df.columns if c.startswith('motivo_')],
    'OHE — privación': [c for c in df.columns if any(c.startswith(p) for p in ['tiene_', 'puede_sustituir_'])],
}

filas = []
for grupo, cols in grupos_gold.items():
    cols_ok = [c for c in cols if c in df.columns]
    if cols_ok:
        filas.append({'Grupo': grupo, 'N variables': len(cols_ok),
                     'Variables': ', '.join(cols_ok[:4]) + ('...' if len(cols_ok) > 4 else '')})

print(pd.DataFrame(filas).to_string(index=False))


---
## G.8 Decisiones de diseño — resumen

| # | Ámbito | Decisión | Justificación |
|---|--------|----------|---------------|
| 1 | Target | Umbral ≥2 de 5 condiciones para `estres_financiero_alto` | Un único indicador puede reflejar una situación puntual; exigir al menos dos garantiza que el estrés sea persistente y multidimensional |
| 2 | Target | NaN en target cuando el score es indeterminable | Con solo 1 componente disponible y score < 2, no es posible determinar la clase con fiabilidad; mejor NaN que ruido |
| 3 | Eliminación | `region`, `situacion_actividad`, `situacion_profesional` eliminadas | Constantes en la muestra (toda es Madrid, asalariados activos); cero varianza, no aportan al modelo |
| 4 | Eliminación | `id_hogar`, `id_persona` eliminados | Son identificadores, no features; incluirlos produciría memorización del dataset |
| 5 | Eliminación | Componentes del target eliminados como features | Causarían *data leakage* directo: el modelo aprendería a predecir el target a partir de sus propias variables de construcción |
| 6 | Eliminación | `renta_hogar_indicadores` eliminada | Correlación = 1.0 con `renta_neta_hogar`; mantenerla solo añade ruido y colinealidad |
| 7 | Feature eng. | `renta_hogar_per_capita` (renta / unidades_consumo) | La renta bruta del hogar no es comparable entre hogares de distinto tamaño; la escala OCDE modificada es el estándar del INE |
| 8 | Feature eng. | `ratio_carga_vivienda` (gastos_vivienda × 12 / renta_neta_salarial) | El porcentaje de renta destinado a vivienda es el indicador europeo estándar de sobrecarga (umbral 30%, Eurostat) |
| 9 | Feature eng. | `precariedad_laboral` (temporal O parcial) | Captura la vulnerabilidad laboral en una sola dimensión; la combinación es más informativa que las dos variables por separado |
| 10 | Feature eng. | Agrupación de `nivel_estudios` en 3 grupos | Dos categorías originales tienen n<20 (`Sin estudios`, `Secundaria 2ª etapa gral`); categorías tan pequeñas generan estimaciones inestables |
| 11 | Feature eng. | `log1p` sobre rentas e importes de vivienda | Skew > 2.5 en todas las variables de renta; la transformación reduce la asimetría y mejora el comportamiento de modelos lineales y basados en distancias |
| 12 | Nulos | `motivo_aumento/disminucion_ingresos` → categoría 'No aplica' | El NaN es semántico: "los ingresos no cambiaron"; crear una categoría explícita preserva esa información sin perder observaciones |
| 13 | Nulos | `expectativa_ingresos_12m` → moda + indicador `expectativa_sin_respuesta` | Es un módulo opcional (9.8% NaN); el indicador binario preserva la información de no-respuesta para el modelo |
| 14 | Nulos | Grupo I (<2%) → mediana (numéricas) o moda (categóricas) | Volumen pequeño, imputación simple y reproducible; en producción se ajusta solo sobre train |
| 15 | Encoding | Encoding ordinal para variables con orden semántico | El orden es información real (peor→mejor salud, mayor→menor dificultad); codificarlo como nominal perdería esa información |
| 16 | Encoding | Encoding binario para variables Sí/No | Convención estándar; evita la expansión de columnas de un OHE innecesario |
| 17 | Encoding | One-hot para variables nominales sin orden | Sector CNAE, tipo de hogar, régimen de tenencia, etc. no tienen orden intrínseco; el OHE respeta esa neutralidad |
| 18 | Desbalanceo | Documentado (1:5.3) pero no corregido en este paso | La corrección (SMOTE, class_weight) pertenece al pipeline de modelado, no al preprocesado de datos, para evitar fuga de información en validación cruzada |
| 19 | Peso muestral | `peso_persona` conservado pero excluido de las features | Se usará para estadísticas descriptivas ponderadas y como argumento `sample_weight` en la evaluación; no es una feature de entrada al modelo |
